# Stable Diffusion 模型结构可视化展示（基于 torchviz）

本 Notebook 使用 `torchviz.make_dot()` 对 Stable Diffusion 2.1-base 的各核心子模块进行**计算图（Computation Graph）可视化**。

## torchviz 与 torchinfo 的区别

| 维度 | torchinfo | torchviz |
|------|-----------|---------- |
| 展示内容 | 模块层级结构（表格） | autograd 计算图（有向无环图）|
| 图中节点 | nn.Module（如 Conv2d、Linear）| Tensor 操作（如 AddmmBackward、ConvolutionBackward）|
| 需要反向传播 | 否（仅前向） | 否，但 Tensor 需保留梯度信息（不能在 no_grad 块中） |
| 直观性 | 高（对齐的表格） | 中等（节点众多时密集）|
| 适合场景 | 分析层结构、参数量、shape | 理解数据流向、反向传播路径、自定义算子调试 |

**覆盖模块：**
| 编号 | 模块 | 输入尺寸（缩小以保持图可读） |
|------|------|--------------------------|
| 三 | VAE 编码器 | `[1, 3, 64, 64]`（缩小 8× 以减少节点数）|
| 四 | VAE 解码器 | `[1, 4, 8, 8]`（缩小 8×）|
| 五 | CLIP 文本编码器 | `[1, 77]` token ids |
| 六 | U-Net 局部子模块 | 时间步嵌入 / conv_in / down_blocks[0] |

## 一、环境准备

### 1.1 安装依赖

`torchviz` 依赖 Python 包 `graphviz` 以及**系统级 Graphviz 工具**（提供 `dot` 可执行文件）。
两者缺一不可，否则 `.render()` 时会报找不到 `dot` 命令的错误。

> **Windows 系统安装 Graphviz 工具：**
> - 推荐方式：`conda install graphviz`（自动配置 PATH）
> - 或手动下载：https://graphviz.org/download/ 并将 `bin/` 目录加入系统 PATH

In [12]:
# !pip install torchviz graphviz  # torchviz: PyTorch 计算图可视化；graphviz: Python 绑定（还需系统级 graphviz 工具）

### 1.2 导入库

In [13]:
import os                                               # 导入 os 模块，用于路径操作和创建输出目录
import torch                                            # 导入 PyTorch，提供张量运算和 autograd 计算图追踪
from torchviz import make_dot                           # 导入 make_dot：输入输出张量 → graphviz Digraph 对象（计算图）
from IPython.display import HTML, display               # 导入 IPython 显示工具：HTML 用于包裹 SVG 并添加尺寸约束，display 触发渲染
from diffusers import AutoencoderKL, UNet2DConditionModel  # 导入 SD 的 VAE 和 U-Net 模型类
from transformers import CLIPTextModel                 # 导入 CLIP 文本编码器类

# 图像输出目录：所有生成的计算图 PNG 文件保存到此目录
OUTPUT_DIR = "./torchviz_output"                       # 输出目录路径（相对于当前工作目录）
os.makedirs(OUTPUT_DIR, exist_ok=True)                 # 创建目录，exist_ok=True 表示已存在时不报错

# 检查设备：优先使用 GPU，否则退回 CPU
# 注意：torchviz 追踪 autograd 图，不依赖具体设备，GPU 和 CPU 均可正常使用
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # DEVICE 类型：str，取值 "cuda" 或 "cpu"
# torchviz 要求输入张量启用梯度（requires_grad=True），以便追踪反向传播路径
# 因此不能在 torch.no_grad() 块中调用 make_dot
# 模型权重：float16（GPU）或 float32（CPU）
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
print(f"当前使用设备：{DEVICE}，权重精度：{DTYPE}")

当前使用设备：cuda，权重精度：torch.float16


### 1.3 加载模型

与 `07.StableDiffusion_torchinfo_Experiment.ipynb` 保持一致，使用 ModelScope 本地缓存加载。

In [14]:
from modelscope import snapshot_download               # 导入 ModelScope 快照下载函数，首次联网下载，后续读取本地缓存

MODEL_ID = "stabilityai/stable-diffusion-2-1-base"    # ModelScope 模型 ID
MODEL_CACHE_DIR = "./model_cache/SD"                   # 本地缓存根目录，与 torchinfo 实验 notebook 保持一致

# snapshot_download 返回 str：本地模型目录绝对路径，首次自动下载，后续直接返回缓存路径
print("正在获取模型本地路径（首次运行时自动下载）...")
model_dir = snapshot_download(MODEL_ID, cache_dir=MODEL_CACHE_DIR)  # model_dir: str，SD 仓库本地根目录
print(f"模型本地路径：{model_dir}")

# 加载 VAE（仅需 encoder 和 decoder 子模块）
vae = AutoencoderKL.from_pretrained(
    model_dir, subfolder="vae",                        # 子目录 "vae" 存放 VAE 权重
    torch_dtype=DTYPE, low_cpu_mem_usage=True,
).to(DEVICE)
vae.eval()                                             # 切换为评估模式：关闭 Dropout 等随机层
print("✓ VAE 加载完成")

# 加载 CLIP 文本编码器
text_encoder = CLIPTextModel.from_pretrained(
    model_dir, subfolder="text_encoder",               # 子目录 "text_encoder" 存放 CLIP 权重
    torch_dtype=DTYPE, low_cpu_mem_usage=True,
).to(DEVICE)
text_encoder.eval()
print("✓ CLIP 文本编码器加载完成")

# 加载 U-Net
unet = UNet2DConditionModel.from_pretrained(
    model_dir, subfolder="unet",                       # 子目录 "unet" 存放 U-Net 权重
    torch_dtype=DTYPE, low_cpu_mem_usage=True,
).to(DEVICE)
unet.eval()
print("✓ U-Net 加载完成")

正在获取模型本地路径（首次运行时自动下载）...


2026-07-21 12:09:40,240 | INFO    | modelscope_hub.download | Downloading 29 files from stabilityai/stable-diffusion-2-1-base@master


Downloading:   0%|          | 0/29 [00:00<?, ?file/s]

模型本地路径：model_cache\SD\models\stabilityai--stable-diffusion-2-1-base\snapshots\master
✓ VAE 加载完成


Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

✓ CLIP 文本编码器加载完成
✓ U-Net 加载完成


## 二、torchviz 核心用法与工具函数

`make_dot(output_tensor, params, show_attrs, show_saved)` 的工作原理：
1. 沿 `output_tensor.grad_fn` 递归遍历 autograd 计算图（反向传播路径）
2. 每个 `grad_fn` 节点（如 `AddmmBackward`、`ConvolutionBackward`）对应一次前向运算
3. 叶子节点为 `params` 中的模型参数（权重/偏置），用矩形框显示
4. 中间节点为临时 Tensor（前向输出），用椭圆显示

**关键约束：**
- 输入张量必须 `requires_grad=True`（否则计算图中断，无法追踪到叶子节点）
- 不能在 `torch.no_grad()` 块内调用（no_grad 会禁用梯度追踪）
- 大模型（如完整 U-Net）的计算图节点极多，需选取局部子模块可视化

> **输出格式说明：** 本 notebook 使用 **SVG 矢量格式**（Scalable Vector Graphics）渲染计算图。
> SVG 与 PNG 光栅格式不同，任意缩放均保持清晰，不会出现放大模糊的问题，特别适合节点密集的计算图展示。

In [15]:
def render_graph(dot, filename: str, display_img: bool = True) -> str:
    """
    将 torchviz 生成的 graphviz Digraph 对象渲染为 SVG 矢量文件，并可选地在 Notebook 内嵌显示。

    SVG（Scalable Vector Graphics）是矢量格式，与 PNG 光栅格式相比，任意缩放均保持清晰，
    不会出现放大模糊的问题，特别适合节点密集的计算图展示。
    显示时通过 HTML 容器添加 max-width 约束，防止图像超出屏幕；缩放仍保持矢量精度。

    参数:
        dot (graphviz.Digraph): make_dot() 返回的计算图对象，包含节点和边的 DOT 描述。
        filename (str): 输出文件名（不含扩展名），最终生成 OUTPUT_DIR/<filename>.svg。
        display_img (bool): 是否在 Notebook 中内嵌显示生成的 SVG 图像，默认 True。

    返回:
        str: 生成的 SVG 文件的完整路径。
    """
    output_path = os.path.join(OUTPUT_DIR, filename)   # 拼接完整输出路径：OUTPUT_DIR/filename（不含后缀）
    dot.render(                                         # 调用 graphviz 渲染引擎将 DOT 描述转换为矢量图文件
        output_path,                                    # 输出文件路径（graphviz 自动追加 .svg 后缀）
        format="svg",                                   # 输出格式：svg 矢量格式，任意缩放不失真
        cleanup=True,                                   # cleanup=True：渲染完成后删除中间 .dot 源文件，只保留 .svg
    )
    svg_path = output_path + ".svg"                    # 拼接最终 SVG 文件路径（graphviz 自动追加 .svg 后缀）
    if display_img:                                     # 若需要在 Notebook 内嵌显示
        with open(svg_path, "r", encoding="utf-8") as f:
            svg_content = f.read()                     # 读取 SVG 文件原始内容（XML 字符串）
        # 用 HTML div 包裹 SVG：max-width:100% 使图像宽度不超过单元格；
        # overflow-x:auto 在图像超宽时显示横向滚动条；height:auto 等比缩放高度
        display(HTML(
            f'<div style="max-width:100%; overflow-x:auto;">'
            f'<div style="display:inline-block; min-width:0;">{svg_content}</div>'
            f'</div>'
        ))
    print(f"计算图已保存：{svg_path}")                 # 打印保存路径，便于定位文件
    return svg_path                                     # 返回 SVG 文件路径，供后续引用或再次显示


def make_graph(model, output_tensor, filename: str, display_img: bool = True):
    """
    对指定模型的输出张量构建计算图并渲染输出。

    make_dot 通过追踪 output_tensor 的 grad_fn 链路自动构建完整的 autograd 计算图，
    并将模型的命名参数（权重/偏置）作为叶子节点标注名称。

    参数:
        model (nn.Module): 参与前向传播的模型，其 named_parameters() 用于标注叶子节点。
        output_tensor (Tensor): 模型前向传播的输出张量，必须有 grad_fn（非 no_grad 环境）。
                                类型：torch.Tensor，须 requires_grad 路径上有可学习参数。
        filename (str): 输出 SVG 文件名（不含扩展名）。
        display_img (bool): 是否在 Notebook 中内嵌显示，默认 True。

    返回:
        str: 生成的 PNG 文件路径。
    """
    dot = make_dot(
        output_tensor,                                  # 输出张量：torchviz 从此张量的 grad_fn 开始反向遍历计算图
        params=dict(model.named_parameters()),          # 命名参数字典：{参数名: Tensor}，用于在图中标注叶子节点名称
        show_attrs=False,                               # show_attrs=False：不显示节点属性（如 shape、stride），保持图简洁
        show_saved=False,                               # show_saved=False：不显示保存的中间张量（减少节点数量）
    )
    dot.graph_attr.update(rankdir="TB")                # TB（Top→Bottom）：图的方向为从上到下（前向传播方向）
    return render_graph(dot, filename, display_img)     # 渲染并返回 SVG 路径

## 三、VAE 计算图可视化

> **输入尺寸说明：** 将标准 512×512 缩小为 64×64（缩小 8×），节点数量从数万减少到可读范围。
> 模型结构完全相同，仅空间分辨率不同，不影响计算图拓扑。

### 3.1 VAE 编码器（Encoder）计算图

In [16]:
print("构建 VAE 编码器计算图...")

# 构造虚拟输入张量
# requires_grad=True：torchviz 需要输入参与 autograd 图，否则只能看到参数叶子节点，看不到数据流路径
# 形状 [batch=1, C=3, H=64, W=64]：缩小版输入，节省节点数量（原 512×512 会产生数万节点）
dummy_img = torch.randn(1, 3, 64, 64, dtype=DTYPE, requires_grad=True).to(DEVICE)

# 前向传播：在 autograd 追踪环境中（不使用 no_grad）执行编码器前向
# vae.encoder(x) 输出形状 [1, 8, 8, 8]（8 通道：前 4 为均值 μ，后 4 为对数方差 log σ²）
encoder_output = vae.encoder(dummy_img)                # encoder_output: Tensor，形状 [1, 8, H/8, W/8]，有 grad_fn

# make_graph 内部调用 make_dot(encoder_output, params=dict(vae.encoder.named_parameters()))
# 从 encoder_output.grad_fn 沿反向路径遍历，构建包含所有 ConvolutionBackward、AddmmBackward 等节点的 DAG
make_graph(vae.encoder, encoder_output, "vae_encoder")

构建 VAE 编码器计算图...


计算图已保存：./torchviz_output\vae_encoder.svg


'./torchviz_output\\vae_encoder.svg'

### 3.2 VAE 解码器（Decoder）计算图

In [17]:
print("构建 VAE 解码器计算图...")

# 解码器接收 4 通道 latent 作为输入（重参数化采样后的 4 通道，非 8 通道）
# 形状 [batch=1, C=4, H=8, W=8]：缩小版 latent（原始 64×64 latent 再缩小 8×）
# requires_grad=True 同样必须设置，保证 autograd 图完整追踪到叶子节点
dummy_latent = torch.randn(1, 4, 8, 8, dtype=DTYPE, requires_grad=True).to(DEVICE)

# vae.decoder(latent) 输出形状 [1, 3, H*8, W*8] = [1, 3, 64, 64]（经 tanh 归一化的 RGB 图像）
decoder_output = vae.decoder(dummy_latent)              # decoder_output: Tensor，形状 [1, 3, 64, 64]，有 grad_fn

make_graph(vae.decoder, decoder_output, "vae_decoder")

构建 VAE 解码器计算图...


计算图已保存：./torchviz_output\vae_decoder.svg


'./torchviz_output\\vae_decoder.svg'

## 四、CLIP 文本编码器计算图

CLIP 文本编码器输入为整数 token id，**整数张量本身不参与 autograd**。
计算图从 `Embedding` 查表后的浮点输出开始追踪，因此图的叶子节点为 Embedding 权重，而非输入 token id。

### 4.1 完整 CLIPTextModel 计算图（23 层 Transformer）

In [18]:
print("构建 CLIP 文本编码器计算图...")

# 输入为整数 token id 序列，dtype=torch.long（int64），不需要 requires_grad
# 形状 [batch=1, seq_len=77]：最大序列长度 77，填充值 49406（BOS token id）
dummy_token_ids = torch.ones(1, 77, dtype=torch.long).to(DEVICE) * 49406

# CLIPTextModel 前向传播：内部 Embedding 查表后进入 Transformer 编码器
# 返回 BaseModelOutputWithPooling，取 .last_hidden_state 作为追踪目标
# last_hidden_state 形状 [1, 77, 1024]：77 个 token 各自的 1024 维语义向量
clip_output = text_encoder(dummy_token_ids).last_hidden_state  # clip_output: Tensor [1, 77, 1024]，有 grad_fn

# 注意：整数输入 dummy_token_ids 本身无 grad_fn，
# 但 Embedding 的权重（token_embedding.weight）是浮点叶子节点，
# 因此 make_dot 仍能从 clip_output 追踪到完整计算图
make_graph(text_encoder, clip_output, "clip_text_encoder")

构建 CLIP 文本编码器计算图...


计算图已保存：./torchviz_output\clip_text_encoder.svg


'./torchviz_output\\clip_text_encoder.svg'

### 4.2 单个 CLIPEncoderLayer 计算图（局部放大）

完整 CLIP 有 23 层 Transformer，计算图节点众多。单独可视化第 0 层可以清晰看到
`LayerNorm → CLIPAttention（Q/K/V/out 投影）→ LayerNorm → MLP（fc1/fc2）` 的计算路径。

In [19]:
from transformers.models.clip.modeling_clip import CLIPEncoderLayer  # 导入 CLIPEncoderLayer 用于 isinstance 匹配

# 动态查找第 0 个 CLIPEncoderLayer（兼容不同版本 Transformers 的属性路径差异）
encoder_layer_0 = None
for layer_name, module in text_encoder.named_modules():  # 遍历所有子模块，layer_name: str，module: nn.Module
    if isinstance(module, CLIPEncoderLayer):            # 找到第一个 CLIPEncoderLayer 实例即为第 0 层
        encoder_layer_0 = module
        print(f"找到 CLIPEncoderLayer，路径：{layer_name}")
        break

# 构造虚拟隐状态张量（模拟 Embedding 层输出）
# requires_grad=True：浮点输入需要梯度，使 autograd 图延伸到输入端（便于观察数据流路径）
# 形状 [1, 77, 1024]：batch=1，seq_len=77，hidden_dim=1024
dummy_hidden = torch.randn(1, 77, 1024, dtype=DTYPE, requires_grad=True).to(DEVICE)

# attention_mask：形状 [1, 1, 77, 77]，全零表示不遮蔽任何 token（CLIP 双向注意力）
# dtype 须与 hidden_states 一致（DTYPE），否则矩阵乘法类型不匹配
dummy_attn_mask = torch.zeros(1, 1, 77, 77, dtype=DTYPE).to(DEVICE)

# 前向传播（该版本 Transformers 中 CLIPEncoderLayer 只需 hidden_states + attention_mask 两个参数）
layer_output = encoder_layer_0(dummy_hidden, dummy_attn_mask)  # layer_output: tuple，取第 0 元素为隐状态 [1, 77, 1024]

# CLIPEncoderLayer 返回 tuple：(hidden_states,) 或 (hidden_states, attn_weights)
# 取第 0 个元素（hidden_states）作为计算图追踪起点
make_graph(encoder_layer_0, layer_output[0], "clip_encoder_layer_0")

找到 CLIPEncoderLayer，路径：encoder.layers.0


计算图已保存：./torchviz_output\clip_encoder_layer_0.svg


'./torchviz_output\\clip_encoder_layer_0.svg'

## 五、U-Net 计算图可视化

完整 U-Net 包含数百万个算子节点，全量可视化意义不大。
按功能分拆为三个局部子图，每个聚焦一个核心组件：

| 子节 | 可视化目标 | 关注点 |
|------|-----------|-------|
| 5.1 | 时间步嵌入（TimestepEmbedding） | 正弦编码 → 两层 MLP 的数据流 |
| 5.2 | conv_in + 时间步注入单个 ResnetBlock | 特征图与时间步嵌入的加法融合路径 |
| 5.3 | CrossAttnDownBlock2D 第一个 BasicTransformerBlock | 自注意力 + 交叉注意力 + FFN 的计算图 |

### 5.1 时间步嵌入（TimestepEmbedding）计算图

`time_proj`（正弦编码）+ `time_embedding`（两层 MLP）将整数时间步 t 转换为 1280 维嵌入向量。
这是 U-Net 感知去噪阶段的核心机制。

In [20]:
print("构建 U-Net 时间步嵌入计算图...")

# time_proj（Timesteps 模块）先将整数时间步转换为正弦余弦位置编码（纯数学运算，无可学习参数）
# 再经过 time_embedding（TimestepEmbedding MLP）升维并非线性变换
# 整数时间步无法参与 autograd，需先经过 time_proj 转为浮点，再跟踪后续 MLP 的计算图

# time_proj 输出：正弦余弦编码，形状 [batch=1, freq_dim=320]，浮点类型
# t 的数值范围 [0, 999]，此处取中间时间步 t=500 作为示例
t_int = torch.tensor([500]).to(DEVICE)                  # 整数时间步，形状 [1]，dtype=torch.long

# unet.time_proj 将整数 t 转换为正弦余弦频率编码，输出浮点张量，形状 [1, 320]
t_emb_sincos = unet.time_proj(t_int).to(DTYPE)          # t_emb_sincos: Tensor [1, 320]，此时尚无 requires_grad

# 对正弦编码启用梯度：time_proj 本身无可学习参数，其输出是 time_embedding MLP 的输入
# 设置 requires_grad=True 使 autograd 图从 time_embedding 的输入开始追踪到输出
t_emb_sincos = t_emb_sincos.detach().requires_grad_(True)  # detach 脱离原计算图，再附加 grad 追踪

# time_embedding（TimestepEmbedding）前向：Linear(320→1280) → SiLU → Linear(1280→1280)
# 输出形状 [1, 1280]：最终时间步嵌入向量，注入 U-Net 各 ResnetBlock
t_emb_out = unet.time_embedding(t_emb_sincos)            # t_emb_out: Tensor [1, 1280]，有 grad_fn

make_graph(unet.time_embedding, t_emb_out, "unet_time_embedding")

构建 U-Net 时间步嵌入计算图...


计算图已保存：./torchviz_output\unet_time_embedding.svg


'./torchviz_output\\unet_time_embedding.svg'

### 5.2 U-Net conv_in + 单个 ResnetBlock 计算图

展示特征图与时间步嵌入通过 `time_emb_proj` Linear 层**加法注入**的计算路径——
这是 Diffusion 模型将时间信息注入空间特征的核心机制。

In [21]:
from diffusers.models.resnet import ResnetBlock2D       # 导入 ResnetBlock2D 用于 isinstance 动态查找

print("构建 U-Net ResnetBlock2D（含时间步注入）计算图...")

# 动态查找 down_blocks 中第一个 ResnetBlock2D（兼容不同 diffusers 版本）
resnet_block = None
for blk_name, blk_module in unet.down_blocks[0].named_modules():  # 遍历第 0 个下采样块的所有子模块
    if isinstance(blk_module, ResnetBlock2D):           # 找到第一个 ResnetBlock2D 实例
        resnet_block = blk_module
        print(f"找到 ResnetBlock2D，路径：down_blocks.0.{blk_name}")
        break

# ResnetBlock2D.forward(hidden_states, temb) 需要两个输入：
# ① hidden_states：特征图，形状 [batch=1, C=320, H=8, W=8]（使用小尺寸 8×8）
# ② temb：时间步嵌入，形状 [batch=1, dim=1280]
# 两者均需 requires_grad=True，以便追踪特征图路径和时间步注入路径
dummy_feat = torch.randn(1, 320, 8, 8, dtype=DTYPE, requires_grad=True).to(DEVICE)    # 特征图 [1, 320, 8, 8]
dummy_temb = torch.randn(1, 1280, dtype=DTYPE, requires_grad=True).to(DEVICE)          # 时间步嵌入 [1, 1280]

# 前向传播：ResnetBlock2D 内部将 temb 经 time_emb_proj 投影后加到特征图上
resnet_output = resnet_block(dummy_feat, dummy_temb)    # resnet_output: Tensor [1, 320, 8, 8]，有 grad_fn

# 创建包含两个输入的参数字典（让计算图同时标注特征图参数和时间步参数的叶子节点）
dot = make_dot(
    resnet_output,                                      # 追踪输出张量的 grad_fn 链路
    params={
        **dict(resnet_block.named_parameters()),        # 模型参数（Conv/GroupNorm/Linear 权重）
        "hidden_states": dummy_feat,                    # 将输入特征图也标注为命名叶子节点
        "temb": dummy_temb,                             # 将时间步嵌入也标注为命名叶子节点
    },
    show_attrs=False,
    show_saved=False,
)
dot.graph_attr.update(rankdir="TB", size="10,10")       # TB 方向，限制图像尺寸
render_graph(dot, "unet_resnet_block_with_temb")

构建 U-Net ResnetBlock2D（含时间步注入）计算图...
找到 ResnetBlock2D，路径：down_blocks.0.resnets.0


计算图已保存：./torchviz_output\unet_resnet_block_with_temb.svg


'./torchviz_output\\unet_resnet_block_with_temb.svg'

### 5.3 BasicTransformerBlock 计算图（自注意力 + 交叉注意力 + FFN）

`BasicTransformerBlock` 是 U-Net 中文本条件注入的核心单元：
- `attn1`（自注意力）：Q/K/V 均来自 latent 特征
- `attn2`（交叉注意力）：Q 来自 latent，K/V 来自 CLIP 文本嵌入
- `ff`（FFN）：GEGLU 门控激活的前馈网络

In [22]:
from diffusers.models.attention import BasicTransformerBlock  # 导入 BasicTransformerBlock 用于动态查找

print("构建 U-Net BasicTransformerBlock 计算图（自注意力 + 交叉注意力）...")

# 动态查找 down_blocks[0] 中第一个 BasicTransformerBlock
transformer_block = None
for blk_name, blk_module in unet.down_blocks[0].named_modules():
    if isinstance(blk_module, BasicTransformerBlock):   # 找到第一个 BasicTransformerBlock
        transformer_block = blk_module
        print(f"找到 BasicTransformerBlock，路径：down_blocks.0.{blk_name}")
        break

# BasicTransformerBlock.forward(hidden_states, encoder_hidden_states=None, ...) 签名
# ① hidden_states：latent 特征（展平后），形状 [batch=1, seq_len=64, dim=320]
#    - seq_len=64 = H×W = 8×8（latent 空间 8×8 展平为序列）
#    - dim=320：down_blocks[0] 的通道数
# ② encoder_hidden_states：CLIP 文本嵌入，形状 [batch=1, text_seq=77, text_dim=1024]
#    - 用于 attn2（交叉注意力）的 K 和 V
dummy_latent_seq = torch.randn(1, 64, 320, dtype=DTYPE, requires_grad=True).to(DEVICE)   # latent 特征序列 [1, 64, 320]
dummy_text_emb = torch.randn(1, 77, 1024, dtype=DTYPE, requires_grad=True).to(DEVICE)    # 文本嵌入 [1, 77, 1024]

# 前向传播：依次经过 norm1→attn1（自注意力）→ norm2→attn2（交叉注意力）→ norm3→ff（GEGLU FFN）
block_output = transformer_block(
    dummy_latent_seq,                                   # hidden_states：latent 特征序列 [1, 64, 320]
    encoder_hidden_states=dummy_text_emb,               # 文本嵌入，作为 Cross-Attention 的 K/V 来源
)                                                       # block_output: Tensor [1, 64, 320]，有 grad_fn

# 同时标注 latent 序列和文本嵌入为命名叶子节点，使计算图清晰显示两条输入路径
dot = make_dot(
    block_output,
    params={
        **dict(transformer_block.named_parameters()),   # 模型参数（Q/K/V/out 投影权重等）
        "hidden_states": dummy_latent_seq,              # latent 输入路径标注
        "encoder_hidden_states": dummy_text_emb,        # 文本嵌入输入路径标注
    },
    show_attrs=False,
    show_saved=False,
)
dot.graph_attr.update(rankdir="TB", size="14,14")       # 稍大尺寸以容纳双输入路径
render_graph(dot, "unet_basic_transformer_block")

构建 U-Net BasicTransformerBlock 计算图（自注意力 + 交叉注意力）...
找到 BasicTransformerBlock，路径：down_blocks.0.attentions.0.transformer_blocks.0


计算图已保存：./torchviz_output\unet_basic_transformer_block.svg


'./torchviz_output\\unet_basic_transformer_block.svg'